In [7]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial import distance
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [8]:
n_community = 2
n_members = 3

tokens = []

for ii in range(n_community*n_members+1):
    tokens.append(
        chr(ord('A')+ii)
    )

In [9]:
class brain(nn.Module):
    def __init__(self, input_size, hidden_wake_size, hidden_sleep_size, sleep_output_size, num_layers=2, num_layers_sleep=2):
        super(brain, self).__init__()

        self.rnn = nn.RNN(input_size+sleep_output_size, hidden_wake_size, num_layers, nonlinearity='relu', batch_first=True)
        self.sleep_rnn = nn.RNN(1, hidden_sleep_size, num_layers_sleep, nonlinearity='relu', batch_first=True)
        self.sleep_fc = nn.Linear(hidden_sleep_size, sleep_output_size)
        self.wake_fc = nn.Linear(hidden_wake_size, len(tokens))
        self.sleep_output_size = sleep_output_size

    def forward(self, x, x_=None, hw=None, hs=None, sleep=False):
        # print(x.shape, 'x')
        if sleep:
            if hs == None:
                out, hs = self.sleep_rnn(x_)
            else:
                out, hs = self.sleep_rnn(x_, hs)
            # print(out.shape)
            sleep_out = self.sleep_fc(out)
        else:
            sleep_out = torch.zeros((1,x.size(1),self.sleep_output_size))
            
        # print(x.size())
        x = torch.cat((x,sleep_out), dim=2)
        
        if hw == None:
            out, hw = self.rnn(x)
        else:
            out, hw = self.rnn(x, hw)

        out = self.wake_fc(out[:,-1,:])

        if sleep:
            return out, hw, hs
        else:
            return out, hw

In [10]:
def compute_geodesic(hidden1, hidden2):

    total_layers = len(hidden1)
    w = 0

    for ii in range(total_layers):
        w_ = np.array(dist( hidden1[ii], hidden2[ii], 'cosine'))
        w += w_
           
    return w[0][0]/total_layers

In [11]:
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=8):
        
        one_hot_encoded = np.zeros((len(data), len(tokens)), dtype=float)
        for ii, token in enumerate(data):
            one_hot_encoded[ii,ord(token)-65] = 1
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory, len(tokens)*working_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), len(tokens)))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                for kk in range(working_memory):
                    self.X[ii,jj,kk*len(tokens):(kk+1)*len(tokens)] = \
                    one_hot_encoded[ii+jj+kk,:]
                    
            self.y[ii] = \
                one_hot_encoded[ii+jj+kk+1,:]

        self.X = tnsr(self.X).float()
        self.y = tnsr(self.y).float()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [12]:
### initial training ###
total_samples = 40000
working_memory = 1
short_term_memory = 1
hidden_wake_size = 35
hidden_sleep_size = 10
sleep_output_size = 5
num_layers_wake = 1
num_layers_sleep = 1
output_sleep = len(tokens)
input_size = len(tokens)*working_memory
lr = 4e-4
test_acc = []

data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

network1 = brain(input_size, hidden_wake_size, hidden_sleep_size, sleep_output_size, num_layers_wake, num_layers_sleep)

optimizer = torch.optim.SGD(network1.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.CrossEntropyLoss()

total = 0
correct = np.zeros(1000,dtype=float)
for X, y in train_loader:
    optimizer.zero_grad()

    if total == 0:
        predicted_y, hidden = network1(X)
    else:
        predicted_y, hidden = network1(X, hw=mem)
    
    # print(predicted_y.shape, y.shape)
    loss = criterion(predicted_y, y)
    loss.backward(retain_graph=True)
    optimizer.step()

    with torch.no_grad():
        mem=hidden.clone()
        true_y = y.argmax(axis=1)
        estimated_y = predicted_y.argmax(axis=1)

        total += 1
        if true_y == estimated_y:
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0

        test_acc.append(
            np.sum(correct)/total if total<1000 else np.sum(correct)/1000
        )
        if total%1000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')

Iter : 1001, loss: 2.0348, accuracy: 0.2440
Iter : 2001, loss: 1.9666, accuracy: 0.2610
Iter : 3001, loss: 1.7629, accuracy: 0.3590
Iter : 4001, loss: 2.5532, accuracy: 0.5050
Iter : 5001, loss: 1.8033, accuracy: 0.5510
Iter : 6001, loss: 2.1082, accuracy: 0.5810
Iter : 7001, loss: 1.8583, accuracy: 0.5970
Iter : 8001, loss: 1.6923, accuracy: 0.5850
Iter : 9001, loss: 0.5288, accuracy: 0.6130
Iter : 10001, loss: 1.3518, accuracy: 0.6520
Iter : 11001, loss: 1.1137, accuracy: 0.6570
Iter : 12001, loss: 1.9556, accuracy: 0.6530
Iter : 13001, loss: 2.1408, accuracy: 0.6440
Iter : 14001, loss: 1.1100, accuracy: 0.6460
Iter : 15001, loss: 2.8234, accuracy: 0.6760
Iter : 16001, loss: 1.2498, accuracy: 0.6680
Iter : 17001, loss: 1.9557, accuracy: 0.6620
Iter : 18001, loss: 1.9754, accuracy: 0.6710
Iter : 19001, loss: 2.0561, accuracy: 0.6520
Iter : 20001, loss: 2.5972, accuracy: 0.6620
Iter : 21001, loss: 1.6642, accuracy: 0.6660
Iter : 22001, loss: 1.7245, accuracy: 0.6740
Iter : 23001, loss:

In [13]:
centroids = []

threshold = .2
n_samples = 1000
idx = torch.randint(0, len(tokens), (1,)) [0]
X_hat = torch.zeros(len(tokens),dtype=torch.float32)
X_hat[idx] = 1.0
counts = []

for ii in range(n_samples):
    if ii == 0:
        # seq += tokens[idx]        
        X_hat, mem = network1(X_hat.reshape(1,1,-1))
        centroids.append(mem[0][0])
        counts.append(1)
    else:
        X_hat, mem = network1(X_hat, mem)

    dis = []
    min_dis = 10
    min_dis_id = -1
    for jj in range(len(centroids)):
        dis.append(
            distance.cosine(centroids[jj].detach().numpy(), mem[0][0].detach().numpy())
        )
        if min_dis >= dis[-1]:
            min_dis = dis[-1] 
            min_dis_id = jj 
    if min_dis < threshold:
        centroids[min_dis_id] = (counts[min_dis_id]*centroids[min_dis_id] + mem[0][0])/(counts[min_dis_id]+1)
        counts[min_dis_id] += 1
    else:
        centroids.append(mem[0][0])
        counts.append(1)

    # print(min_dis)   
    X_hat = torch.nn.functional.softmax(X_hat, dim=1)
    dist_categ = torch.distributions.Categorical(probs=X_hat.reshape(-1))
    idx = dist_categ.sample()

    X_hat = torch.zeros(len(tokens),dtype=torch.float32)
    X_hat[idx] = 1.0
    X_hat = X_hat.reshape(1,1,-1)  

    # for ii, center in enumerate(centroids): 
    #     centroids[ii] = centroids[ii].detach().numpy()

In [14]:
mega_cluster = []
threshold = .6
labels = [0]
total_centroids = len(centroids)

current_label = 0
flag = True
for ii in range(1,total_centroids):
    for jj in range(ii-1,-1,-1):
        # print(distance.cosine(centroids[ii].detach().numpy(), centroids[jj].detach().numpy()), ii, jj)
        if distance.cosine(centroids[ii].detach().numpy(), centroids[jj].detach().numpy()) <= threshold:
            labels.append(labels[jj])
            flag = False
            break 
        
    if flag :
        current_label += 1
        labels.append(current_label)
    flag = True 
    

In [15]:
labels

[0, 0, 0, 1, 1, 1, 1]

In [309]:
centroids

[tensor([0.0000, 0.2648, 0.0000, 0.0000, 0.8129, 0.0000, 0.8000, 0.2769, 0.0000,
         0.0000, 1.2565, 0.7098, 0.0000, 1.1055, 1.1179, 0.0000, 0.2215, 0.1823,
         0.2993, 0.1863, 0.8149, 0.4746, 0.0790, 0.1466, 0.0000, 0.0000, 0.8307,
         0.0000, 0.8197, 0.0000, 0.3933, 0.0000, 0.8320, 0.0000, 0.0000],
        grad_fn=<DivBackward0>),
 tensor([0.0000, 0.0000, 0.9337, 0.0000, 0.3168, 0.0000, 0.8134, 0.0000, 0.0000,
         0.3541, 0.9357, 0.4128, 0.0000, 0.0000, 1.3000, 0.0000, 0.1376, 1.6718,
         0.7045, 0.0000, 0.8344, 0.5734, 0.2391, 0.0000, 0.0969, 0.0000, 0.0000,
         0.0000, 0.3952, 0.4628, 0.0000, 0.2883, 0.2417, 0.0000, 0.1294],
        grad_fn=<DivBackward0>),
 tensor([0.0000, 0.0000, 0.8662, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.5627, 1.4597, 1.1423, 0.4705, 1.1326, 0.0000, 0.0000, 0.4549,
         0.0000, 0.0000, 0.4096, 0.1485, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.3203, 0.0000, 0.0000, 0.8779, 0.6199, 

In [310]:
n_samples = 1000
data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

community = {}
for ii in range(len(centroids)):
    community[ii] = []

ii = 0

for X, y in train_loader:
    if ii == 0:
        # seq += tokens[idx]        
        X_hat, mem = network1(X.reshape(1,1,-1))
        # centroids.append(mem[0][0])
    else:
        X_hat, mem = network1(X, mem)
    
    dis = []
    for center in centroids:
        dis.append(
            distance.cosine(center.detach().numpy(), mem[0][0].detach().numpy())
        )
    
    idx = np.argmin(dis)
    community[idx].append(tokens[X.argmax(axis=2)])

         
    ii += 1

In [311]:
for com in community.keys():
    print(np.unique(community[com]))

['B']
['A']
['C']
['E']
['F']
['D']
['G']


In [312]:
centroids_ = []
for ii, center in enumerate(centroids): 
    centroids_.append(centroids[ii].detach().numpy())

In [315]:


model_compressor = LogisticRegression(multi_class='ovr', solver='lbfgs')
model_compressor.fit(centroids_, labels)


/opt/anaconda3/envs/efri/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression(multi_class='ovr')

In [316]:
model_compressor.predict(centroids_)

array([0, 0, 0, 1, 1, 1, 1])